# 05 — Cnn Lstm Forecast Model / 卷积神经网络

**Chapter 14 — File 5 of 6 / 第14章 — 第5个文件（共6个）**

---

## Summary / 总结

This script demonstrates **evaluate cnn-lstm for monthly car sales dataset**.

本脚本演示 **evaluate cnn-lstm for monthly car sales dataset**。

---
## Step 1 — evaluate cnn-lstm for monthly car sales dataset

In [ ]:
from math import sqrt
from numpy import array
from numpy import mean
from numpy import std
from pandas import DataFrame
from pandas import concat
from pandas import read_csv
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import TimeDistributed
from keras.layers import Flatten
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from matplotlib import pyplot

---
## Step 2 — split a univariate dataset into train/test sets

In [ ]:
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

---
## Step 3 — transform list into supervised learning format

In [ ]:
def series_to_supervised(data, n_in, n_out=1):
	df = DataFrame(data)
	cols = list()

---
## Step 4 — input sequence (t-n, ... t-1)

In [ ]:
for i in range(n_in, 0, -1):
		cols.append(df.shift(i))

---
## Step 5 — forecast sequence (t, t+1, ... t+n)

In [ ]:
for i in range(0, n_out):
		cols.append(df.shift(-i))

---
## Step 6 — put it all together

In [ ]:
agg = concat(cols, axis=1)

---
## Step 7 — drop rows with NaN values

In [ ]:
agg.dropna(inplace=True)
	return agg.values

---
## Step 8 — root mean squared error or rmse

In [ ]:
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

---
## Step 9 — fit a model

In [ ]:
def model_fit(train, config):

---
## Step 10 — unpack config

In [ ]:
n_seq, n_steps, n_filters, n_kernel, n_nodes, n_epochs, n_batch = config
	n_input = n_seq * n_steps

---
## Step 11 — prepare data

In [ ]:
data = series_to_supervised(train, n_input)
	train_x, train_y = data[:, :-1], data[:, -1]
	train_x = train_x.reshape((train_x.shape[0], n_seq, n_steps, 1))

---
## Step 12 — define model

In [ ]:
model = Sequential()
	model.add(TimeDistributed(Conv1D(n_filters, n_kernel, activation='relu', input_shape=(None,n_steps,1))))
	model.add(TimeDistributed(Conv1D(n_filters, n_kernel, activation='relu')))
	model.add(TimeDistributed(MaxPooling1D()))
	model.add(TimeDistributed(Flatten()))
	model.add(LSTM(n_nodes, activation='relu'))
	model.add(Dense(n_nodes, activation='relu'))
	model.add(Dense(1))
	model.compile(loss='mse', optimizer='adam')

---
## Step 13 — fit

In [ ]:
model.fit(train_x, train_y, epochs=n_epochs, batch_size=n_batch, verbose=0)
	return model

---
## Step 14 — forecast with a pre-fit model

In [ ]:
def model_predict(model, history, config):

---
## Step 15 — unpack config

In [ ]:
n_seq, n_steps, _, _, _, _, _ = config
	n_input = n_seq * n_steps

---
## Step 16 — prepare data

In [ ]:
x_input = array(history[-n_input:]).reshape((1, n_seq, n_steps, 1))

---
## Step 17 — forecast

In [ ]:
yhat = model.predict(x_input, verbose=0)
	return yhat[0]

---
## Step 18 — walk-forward validation for univariate data

In [ ]:
def walk_forward_validation(data, n_test, cfg):
	predictions = list()

---
## Step 19 — split dataset

In [ ]:
train, test = train_test_split(data, n_test)

---
## Step 20 — fit model

In [ ]:
model = model_fit(train, cfg)

---
## Step 21 — seed history with training dataset

In [ ]:
history = [x for x in train]

---
## Step 22 — step over each time-step in the test set

In [ ]:
for i in range(len(test)):

---
## Step 23 — fit model and make forecast for history

In [ ]:
yhat = model_predict(model, history, cfg)

---
## Step 24 — store forecast in list of predictions

In [ ]:
predictions.append(yhat)

---
## Step 25 — add actual observation to history for the next loop

In [ ]:
history.append(test[i])

---
## Step 26 — estimate prediction error

In [ ]:
error = measure_rmse(test, predictions)
	print(' > %.3f' % error)
	return error

---
## Step 27 — repeat evaluation of a config

In [ ]:
def repeat_evaluate(data, config, n_test, n_repeats=30):

---
## Step 28 — fit and evaluate the model n times

In [ ]:
scores = [walk_forward_validation(data, n_test, config) for _ in range(n_repeats)]
	return scores

---
## Step 29 — summarize model performance

In [ ]:
def summarize_scores(name, scores):

---
## Step 30 — print a summary

In [ ]:
scores_m, score_std = mean(scores), std(scores)
	print('%s: %.3f RMSE (+/- %.3f)' % (name, scores_m, score_std))

---
## Step 31 — box and whisker plot

In [ ]:
pyplot.boxplot(scores)
	pyplot.show()

series = read_csv('monthly-car-sales.csv', header=0, index_col=0)
data = series.values

---
## Step 32 — data split

In [ ]:
n_test = 12

---
## Step 33 — define config

In [ ]:
config = [3, 12, 64, 3, 100, 200, 100]

---
## Step 34 — grid search

In [ ]:
scores = repeat_evaluate(data, config, n_test)

---
## Step 35 — summarize scores

In [ ]:
summarize_scores('cnn-lstm', scores)

---
## Learning Notes / 学习笔记

- **概念**: evaluate cnn-lstm for monthly car sales dataset 是机器学习中的常用技术。  
  *evaluate cnn-lstm for monthly car sales dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Cnn Lstm Forecast Model / 卷积神经网络
# Complete Code / 完整代码
# ===============================

# evaluate cnn-lstm for monthly car sales dataset
from math import sqrt
from numpy import array
from numpy import mean
from numpy import std
from pandas import DataFrame
from pandas import concat
from pandas import read_csv
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import TimeDistributed
from keras.layers import Flatten
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from matplotlib import pyplot

# split a univariate dataset into train/test sets
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

# transform list into supervised learning format
def series_to_supervised(data, n_in, n_out=1):
	df = DataFrame(data)
	cols = list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
	# put it all together
	agg = concat(cols, axis=1)
	# drop rows with NaN values
	agg.dropna(inplace=True)
	return agg.values

# root mean squared error or rmse
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

# fit a model
def model_fit(train, config):
	# unpack config
	n_seq, n_steps, n_filters, n_kernel, n_nodes, n_epochs, n_batch = config
	n_input = n_seq * n_steps
	# prepare data
	data = series_to_supervised(train, n_input)
	train_x, train_y = data[:, :-1], data[:, -1]
	train_x = train_x.reshape((train_x.shape[0], n_seq, n_steps, 1))
	# define model
	model = Sequential()
	model.add(TimeDistributed(Conv1D(n_filters, n_kernel, activation='relu', input_shape=(None,n_steps,1))))
	model.add(TimeDistributed(Conv1D(n_filters, n_kernel, activation='relu')))
	model.add(TimeDistributed(MaxPooling1D()))
	model.add(TimeDistributed(Flatten()))
	model.add(LSTM(n_nodes, activation='relu'))
	model.add(Dense(n_nodes, activation='relu'))
	model.add(Dense(1))
	model.compile(loss='mse', optimizer='adam')
	# fit
	model.fit(train_x, train_y, epochs=n_epochs, batch_size=n_batch, verbose=0)
	return model

# forecast with a pre-fit model
def model_predict(model, history, config):
	# unpack config
	n_seq, n_steps, _, _, _, _, _ = config
	n_input = n_seq * n_steps
	# prepare data
	x_input = array(history[-n_input:]).reshape((1, n_seq, n_steps, 1))
	# forecast
	yhat = model.predict(x_input, verbose=0)
	return yhat[0]

# walk-forward validation for univariate data
def walk_forward_validation(data, n_test, cfg):
	predictions = list()
	# split dataset
	train, test = train_test_split(data, n_test)
	# fit model
	model = model_fit(train, cfg)
	# seed history with training dataset
	history = [x for x in train]
	# step over each time-step in the test set
	for i in range(len(test)):
		# fit model and make forecast for history
		yhat = model_predict(model, history, cfg)
		# store forecast in list of predictions
		predictions.append(yhat)
		# add actual observation to history for the next loop
		history.append(test[i])
	# estimate prediction error
	error = measure_rmse(test, predictions)
	print(' > %.3f' % error)
	return error

# repeat evaluation of a config
def repeat_evaluate(data, config, n_test, n_repeats=30):
	# fit and evaluate the model n times
	scores = [walk_forward_validation(data, n_test, config) for _ in range(n_repeats)]
	return scores

# summarize model performance
def summarize_scores(name, scores):
	# print a summary
	scores_m, score_std = mean(scores), std(scores)
	print('%s: %.3f RMSE (+/- %.3f)' % (name, scores_m, score_std))
	# box and whisker plot
	pyplot.boxplot(scores)
	pyplot.show()

series = read_csv('monthly-car-sales.csv', header=0, index_col=0)
data = series.values
# data split
n_test = 12
# define config
config = [3, 12, 64, 3, 100, 200, 100]
# grid search
scores = repeat_evaluate(data, config, n_test)
# summarize scores
summarize_scores('cnn-lstm', scores)

---

➡️ **Next / 下一步**: File 6 of 6